# EDA Profile

Auto-EDA profiles (ydata-profiling), one HTML **and** PDF each (PDF via headless Chrome), regenerated on every run:
- `eda/eda-raw/` — the raw source tables
- `eda/eda-interim/` — the engineered feature tables (`data/interim/*.pkl`)
- `eda/eda-master/` — the assembled modelling master

Large tables sampled to 100k rows.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import subprocess
import pandas as pd
from ydata_profiling import ProfileReport
from pathlib import Path

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")
EDA = Path("../eda")
SAMPLE = 100_000
CHROME = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"

def report(df, name, outdir):
    outdir.mkdir(parents=True, exist_ok=True)
    if len(df) > SAMPLE:
        df = df.sample(SAMPLE, random_state=0)
    html = outdir / f"{name}.html"
    ProfileReport(df, title=name, minimal=True).to_file(html)
    subprocess.run([CHROME, "--headless=new", "--disable-gpu", "--no-pdf-header-footer",
                    "--virtual-time-budget=60000",
                    f"--print-to-pdf={html.with_suffix('.pdf')}", html.resolve().as_uri()],
                   check=True, capture_output=True)

## Raw

In [2]:
for name in ["application_train", "bureau", "bureau_balance", "previous_application",
             "POS_CASH_balance", "installments_payments", "credit_card_balance"]:
    report(pd.read_csv(RAW / f"{name}.csv"), name, EDA / "eda-raw")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 706.23it/s]


## Interim

In [3]:
for pkl in sorted(INTERIM.glob("*.pkl")):
    report(pd.read_pickle(pkl).drop(columns="SK_ID_CURR", errors="ignore"), pkl.stem, EDA / "eda-interim")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 101.59it/s]


## Master

In [4]:
from src.data import load_master
X, y = load_master()
report(X.assign(TARGET=y.values), "master", EDA / "eda-master")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 17.72it/s]
